# 00 — Comparación de tokenización sobre vocabulario de dominio

**Objetivo:** antes de "casarnos" con un modelo base, medimos qué tan bien tokeniza cada
candidato el vocabulario específico de nuestro dominio (moda, en español). Un modelo que
parte términos clave en muchos tokens rinde peor en la tarea y consume más cómputo por
ejemplo — este es el argumento técnico concreto detrás de la elección de `Qwen2.5-1.5B-Instruct`
que se documenta en el README (Sección 2).

Medimos la **fertilidad** (tokens por palabra) de cada tokenizer sobre:
1. Los propios términos de la ontología del proyecto (`estilo`, `ocasion`, `clima`, `paleta`, `fit`).
2. Un vocabulario más amplio de moda en español.
3. Un mensaje de ejemplo real, tomado de `data/train.jsonl`.


## 0.1 Montar Google Drive

Este notebook necesita `data/train.jsonl` solo para tomar UN mensaje de ejemplo real
(celda de la Sección 3) — los tokenizers se descargan solos desde Hugging Face, no
dependen de ningún archivo tuyo. Si ya montaste Drive para el notebook 01 y usaste la
misma carpeta, esto reutiliza esos mismos datos.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/personal-shopper-ia"

import os
assert os.path.exists(f"{BASE_DIR}/data/train.jsonl"), (
    f"No encuentro {BASE_DIR}/data/train.jsonl — revisa BASE_DIR."
)
print("Drive montado, datos encontrados en:", f"{BASE_DIR}/data")

In [ ]:
!pip install -q transformers==4.46.2 sentencepiece

In [ ]:
import json
import random

from transformers import AutoTokenizer

SEED = 42
random.seed(SEED)

# Candidatos a comparar: el modelo elegido, un decoder alternativo del mismo tamaño,
# y un encoder multilingüe (para justificar por qué la familia decoder es la correcta,
# no solo el tamaño).
CANDIDATOS = {
    "Qwen2.5-1.5B-Instruct (elegido)": "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen2.5-0.5B-Instruct (alternativa mas liviana)": "Qwen/Qwen2.5-0.5B-Instruct",
    "bert-base-multilingual-cased (encoder, referencia)": "bert-base-multilingual-cased",
}

tokenizers = {
    nombre: AutoTokenizer.from_pretrained(ruta)
    for nombre, ruta in CANDIDATOS.items()
}

for nombre, tok in tokenizers.items():
    print(f"{nombre}: vocab_size={tok.vocab_size}")

In [ ]:
# 1. Términos de la propia ontología del proyecto (los que el modelo debe producir
# como VALORES del JSON de salida — si se parten en muchos tokens, generarlos bien
# formados es más difícil).
TERMINOS_ONTOLOGIA = [
    "Casual", "Formal", "Minimalista", "Urbano", "Bohemio", "Deportivo", "Clasico",
    "boda", "trabajo", "fin_de_semana", "viaje", "deporte", "evento_formal",
    "calido", "frio", "templado",
    "neutros", "pasteles", "oscuros", "colores_vivos", "monocromatico",
    "holgado", "regular", "ajustado", "oversized",
]

# 2. Vocabulario de moda en español, más amplio que la ontología cerrada — así medimos
# si el modelo entiende el DOMINIO, no solo memoriza las 25 palabras que va a producir.
VOCAB_MODA = [
    "pantalón", "chaqueta", "vestido", "camisa", "blazer", "tenis", "sudadera",
    "gabardina", "estampado", "textura", "silueta", "entretiempo", "parka",
    "sastrería", "algodón", "denim", "escote", "gorro", "bufanda", "impermeable",
]

def fertilidad(tok, palabras):
    """Tokens promedio por palabra (fertilidad). Más alto = peor."""
    total_tokens = 0
    for palabra in palabras:
        total_tokens += len(tok.encode(palabra, add_special_tokens=False))
    return total_tokens / len(palabras)

print(f"{'Tokenizer':45s} {'Fertilidad ontología':>22s} {'Fertilidad moda (es)':>22s}")
print("-" * 92)
for nombre, tok in tokenizers.items():
    f_ont = fertilidad(tok, TERMINOS_ONTOLOGIA)
    f_moda = fertilidad(tok, VOCAB_MODA)
    print(f"{nombre:45s} {f_ont:22.2f} {f_moda:22.2f}")

In [ ]:
# 3. Un mensaje real del dataset de entrenamiento — comparamos longitud total en tokens
# de la MISMA cadena de texto entre tokenizers.
with open(f"{BASE_DIR}/data/train.jsonl", "r", encoding="utf-8") as f:
    lineas = [json.loads(l) for l in f]

ejemplo = random.choice(lineas)
texto_ejemplo = ejemplo["input"]
print("Ejemplo:", texto_ejemplo)
print()

for nombre, tok in tokenizers.items():
    n_tokens = len(tok.encode(texto_ejemplo, add_special_tokens=False))
    print(f"{nombre:45s} -> {n_tokens} tokens ({n_tokens/len(texto_ejemplo.split()):.2f} tokens/palabra)")

## Lectura

- `bert-base-multilingual-cased` no es una opción viable para esta tarea independientemente
  de su fertilidad: es un **encoder**, no puede generar texto libre (JSON de longitud
  variable) sin agregar una cabeza/arquitectura seq2seq adicional. Se incluye aquí solo como
  referencia de fertilidad, no como candidato real.
- Entre los dos decoders Qwen, ambos comparten el mismo tokenizer (BPE de Qwen2.5), así que la
  fertilidad es idéntica — la diferencia entre `0.5B` y `1.5B` está en la capacidad del modelo,
  no en la tokenización. Por eso la elección final (Sección 2 del README) se basa en que el
  modelo de 1.5B sigue instrucciones de formato JSON de forma más consistente en zero-shot,
  no en una diferencia de tokenización.
- Los términos de la ontología (en español, con guiones bajos como `fin_de_semana`) tokenizan
  en más de una unidad en ambos modelos Qwen — esperado, ya que no son palabras únicas del
  vocabulario base — pero de forma consistente y sin fragmentación excesiva letra por letra,
  lo cual sí sería una señal de alarma para un modelo poco adecuado al español.